# Model Comparison

## Overview

In the previous notebook, we established a baseline by training and evaluating a Linear Regression model using a train-validation split. While this provided an initial understanding of the model's predictive performance, the evaluation was based on a single validation split, which may not provide a reliable estimate of the model's generalization performance.

In this notebook, we compare Linear Regression and three regularized regression models using **5-fold Cross-Validation**. Unlike a single train-validation split, cross-validation evaluates each model across multiple validation folds, reducing the dependence on any particular data split and providing a more robust estimate of model performance.

The models compared in this notebook are:

- Linear Regression
- Ridge Regression (L2 Regularization)
- Lasso Regression (L1 Regularization)
- ElasticNet Regression (L1 + L2 Regularization)

Each model is evaluated using the same preprocessing pipeline, cross-validation strategy, and evaluation metrics to ensure a fair comparison.

---

## Objectives

The objectives of this notebook are to:

- Evaluate multiple linear regression models using 5-fold Cross-Validation.
- Compare model performance using consistent evaluation metrics.
- Analyze the impact of different regularization techniques on model generalization.
- Identify the most promising model(s) for hyperparameter tuning.
---

## Notebook Workflow

```text
Initialize Model
        │
        ▼
Perform 5-Fold Cross-Validation
        │
        ▼
Compute Mean Performance
        │
        ▼
Interpret Results
        │
        ▼
Compare Models
        │
        ▼
Select Model(s) for Hyperparameter Tuning
```

---

## Evaluation Metrics

Each model is evaluated using the following metrics:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Coefficient of Determination (R²)

For each metric, the **mean** and **standard deviation** across the five cross-validation folds are reported. The mean provides an estimate of the model's average performance, while the standard deviation indicates the consistency of that performance across different validation folds.

---

## Expected Outcome

By the end of this notebook, we will:

- Compare the cross-validation performance of Linear Regression, Ridge Regression, Lasso Regression, and ElasticNet Regression.
- Assess the stability and generalization performance of each model.
- Select the most suitable model(s) for hyperparameter tuning in the next stage of the project.

---

In [35]:
#import libraries

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

#set parh for access src module
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

#import resusable utility 
from src.pipeline import create_pipeline
from src.evaluation import evaluate_regression_model


#pandas display setting 
pd.options.display.float_format = "{:.4f}".format

In [36]:
#load data
df=pd.read_csv('../data/train.csv')

## 1.1 Cross-Validation Strategy

Before comparing the regression models, a **5-fold Cross-Validation** strategy is adopted to obtain a more reliable estimate of each model's generalization performance.

In a single train-validation split, the evaluation metrics may vary depending on how the dataset is partitioned. As a result, the observed performance may not accurately represent the model's ability to generalize to unseen data.

To reduce this dependence on a single split, **5-fold Cross-Validation** divides the training dataset into five approximately equal-sized folds. During each iteration, four folds are used to train the model, while the remaining fold is used for validation. This process is repeated five times, allowing every observation to serve as the validation set exactly once.

The evaluation metrics obtained from the five validation folds are then averaged to provide a more robust estimate of model performance.

For this project:

- **5-fold Cross-Validation** is used.
- The same cross-validation strategy is applied to every model.
- The mean and standard deviation of each evaluation metric are reported for comparison.

In [37]:

# Separate Features and Target Variable
X = df.drop(columns=["SalePrice",])
y = df["SalePrice"]

# Train-Validation Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=40
)

In [38]:
print(f"Training Features   : {X_train.shape}")
print(f"Validation Features : {X_valid.shape}")

print()

print(f"Training Target     : {y_train.shape}")
print(f"Validation Target   : {y_valid.shape}")

Training Features   : (934, 79)
Validation Features : (234, 79)

Training Target     : (934,)
Validation Target   : (234,)


----

## Baseline Model

> **Note:**  
> The baseline Linear Regression model is retrained in this notebook using the same train-validation split as the regularized models. This ensures that all models are evaluated under identical conditions, enabling a fair performance comparison. The implementation and detailed evaluation of the baseline model are presented in **Notebook 06: Baseline Model Evaluation**.

In [39]:
#Initialize and train pipeline
baseline_model_pipeline = create_pipeline(
    LinearRegression()
)

baseline_model_pipeline.fit(X_train, y_train)



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dataset_preprocessing', ...), ('column_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('numerical_missing_values', ...), ('categorical_missing_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
lotfrontage_median_,"Series[float64](25,)",Neighborhood ...dtype: float64
Name,Type,Value


In [6]:
#generate predictions
y_train_pred = baseline_model_pipeline.predict(X_train)

y_valid_pred = baseline_model_pipeline.predict(X_valid)

In [7]:
# Number of features after preprocessing
n_features = len(
    baseline_model_pipeline.named_steps["column_transformer"]
    .get_feature_names_out()
)

# Training metrics
baseline_train_metrics = evaluate_regression_model(
    y_true=y_train,
    y_pred=y_train_pred,
    n_features=n_features,
    adjusted_r2=True,
)
# Validation metrics
baseline_validation_metrics = evaluate_regression_model(
    y_true=y_valid,
    y_pred=y_valid_pred,
)

In [8]:
#summary
baseline_model_evaluation_summary = pd.DataFrame(
    {
        "Training": baseline_train_metrics,
        "Validation": baseline_validation_metrics,
    }
).round(4)

baseline_model_evaluation_summary

,Training,Validation
MAE,11982.7077,19473.3362
MSE,307728957.4972,1511070187.3535
RMSE,17542.2050,38872.4863
R²,0.9522,0.7898
Adjusted R²,0.9302,NaN


##  1.  Ridge Regression

### Overview

Ridge Regression is a regularized version of Linear Regression that incorporates an **L2 regularization** term into the loss function. By penalizing large coefficient values during training, it reduces model complexity and helps improve generalization on unseen data.

Unlike Linear Regression, Ridge Regression retains all input features while shrinking their coefficients toward zero. This makes it particularly useful when the model exhibits overfitting or when predictor variables are highly correlated.

In this section, we train a Ridge Regression model using the same preprocessed dataset as the baseline model and evaluate its performance using the same metrics for a fair comparison.

---

### Objectives

The objectives of this section are to:

- Train a Ridge Regression model using the preprocessed training dataset.
- Generate predictions for the training and validation datasets.
- Evaluate model performance using the reusable evaluation function.
- Compare its performance with the baseline Linear Regression model.
---

### 1.1 Model Initialization

The Ridge Regression model is initialized using the `Ridge` estimator from scikit-learn. For this initial comparison, the model is created with its default hyperparameters. This establishes a baseline for Ridge Regression, allowing its performance to be evaluated before exploring hyperparameter tuning in a later notebook. 

The Ridge Regression model is initialized as part of a reusable machine learning pipeline that combines data preprocessing and model training into a single workflow

In [9]:
from sklearn.linear_model import Ridge

ridge_pipeline = create_pipeline(Ridge())

---

### 1.2 Model Training

The initialized pipeline is trained using the training dataset. During training, the preprocessing pipeline learns the required data transformations, and the Ridge Regression model learns the relationship between the input features and the target variable while applying L2 regularization.

Training the complete pipeline ensures that all preprocessing steps and model fitting are performed as a single workflow, reducing the risk of inconsistencies between training and prediction.

In [10]:
ridge_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dataset_preprocessing', ...), ('column_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('numerical_missing_values', ...), ('categorical_missing_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
lotfrontage_median_,"Series[float64](25,)",Neighborhood ...dtype: float64
Name,Type,Value


---

### 1.3 Generate Predictions

Once the pipeline has been trained, it is used to generate predictions for both the training and validation datasets. These predictions are used to evaluate the model's performance and assess its ability to generalize to unseen data.


In [11]:
y_train_pred = ridge_pipeline.predict(X_train)
y_valid_pred = ridge_pipeline.predict(X_valid)

----

### 1.4 Model Evaluation

The performance of the trained Ridge Regression model is evaluated on both the training and validation datasets using the reusable regression evaluation function developed earlier in the project. The same evaluation metrics used for the baseline Linear Regression model are applied to ensure a consistent and fair comparison.

The following metrics are computed:

- Mean Absolute Error (MAE)
- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Coefficient of Determination (R²)
- Adjusted R² *(Training dataset only)*


In [12]:

# Number of features after preprocessing
n_features = len(
    ridge_pipeline.named_steps["column_transformer"]
    .get_feature_names_out()
)

# Training metrics
ridge_train_metrics = evaluate_regression_model(
    y_true=y_train,
    y_pred=y_train_pred,
    n_features=n_features,
    adjusted_r2=True,
)
# Validation metrics
ridge_validation_metrics = evaluate_regression_model(
    y_true=y_valid,
    y_pred=y_valid_pred,
)

----

### 1.5 Perfomance Summary

In [13]:
ridge_model_evaluation_summary = pd.DataFrame(
    {
        "Training": ridge_train_metrics,
        "Validation": ridge_validation_metrics,
    }
)

ridge_model_evaluation_summary

,Training,Validation
MAE,13850.4734,18882.8172
MSE,431200863.5344,1132045883.0298
RMSE,20765.3766,33645.8895
R²,0.9331,0.8425
Adjusted R²,0.9023,NaN


### cross checking for any bugs 

In [14]:
ridge_pipeline.named_steps["model"].alpha

1.0

In [15]:
X_train_transformed = ridge_pipeline[:-1].fit_transform(X_train, y_train)

print(X_train_transformed.shape)

(934, 294)


In [16]:
baseline_coef = np.abs(
    baseline_model_pipeline.named_steps["model"].coef_
).mean()

ridge_coef = np.abs(
    ridge_pipeline.named_steps["model"].coef_
).mean()

print(baseline_coef)
print(ridge_coef)

15715.487241941235
7776.602350110502


----

### 1.6 Interpretation

The Ridge Regression model demonstrates a notable improvement in validation performance compared with the baseline Linear Regression model.

Although the training R² decreased slightly from **0.9401** to **0.9247**, the validation R² increased substantially from **0.4430** to **0.8846**. Similarly, the validation error metrics (MAE, MSE, and RMSE) decreased considerably.

The small reduction in training performance indicates that the model is less closely fitted to the training data. However, the significant improvement on the validation dataset suggests that the model generalizes much better to unseen data.

This improvement is expected because Ridge Regression applies L2 regularization, which penalizes large coefficient values during training. The regularization term reduces model complexity and helps prevent overfitting while retaining all input features.

Overall, Ridge Regression achieves a better balance between fitting the training data and generalizing to new observations, making it a stronger candidate than the baseline Linear Regression model.

-----

## 2. Lasso Regression

### Overview

Lasso (Least Absolute Shrinkage and Selection Operator) Regression is a regularized version of Linear Regression that incorporates an **L1 regularization** term into the loss function. By penalizing the absolute values of the model coefficients, Lasso reduces model complexity and can shrink some coefficients exactly to zero, effectively performing automatic feature selection.

This characteristic makes Lasso particularly useful when the dataset contains irrelevant or redundant features. In this section, we train a Lasso Regression model using the same preprocessing pipeline and evaluate its performance using the same metrics as the previous models for a fair comparison.

---

### Objectives

The objectives of this section are to:

- Train a Lasso Regression model using the preprocessed training dataset.
- Generate predictions for the training and validation datasets.
- Evaluate model performance using the reusable evaluation function.
- Compare its performance with the baseline Linear Regression and Ridge Regression models.

---

### 2.1 Model Initialization

The Lasso Regression model is initialized using the `Lasso` estimator from scikit-learn with its default hyperparameters. To maintain a consistent workflow, the model is integrated into the reusable machine learning pipeline developed earlier in the project.

Using the same preprocessing pipeline for all models ensures that each model is trained and evaluated under identical preprocessing steps, enabling a fair comparison.





In [17]:
from sklearn.linear_model import Lasso

lasso_pipeline = create_pipeline(
    Lasso(max_iter=50000)
)

lasso_pipeline.named_steps["model"]

,"max_iter max_iter: int, default=1000The maximum number of iterations.",50000
,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


----

### 2.2 Model Training

The initialized pipeline is trained using the training dataset. During training, the preprocessing pipeline learns the required data transformations, and the Lasso Regression model learns the relationship between the input features and the target variable while applying L1 regularization.

Training the complete pipeline ensures that all preprocessing steps and model fitting are performed as a single workflow, reducing the risk of inconsistencies between training and prediction.

In [18]:
lasso_pipeline.fit(X_train, y_train)

C:\Users\ASUS\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_coordinate_descent.py:840: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.315018e+09, tolerance: 6.016e+08
  model = cd_fast.enet_coordinate_descent(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dataset_preprocessing', ...), ('column_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('numerical_missing_values', ...), ('categorical_missing_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
lotfrontage_median_,"Series[float64](25,)",Neighborhood ...dtype: float64
Name,Type,Value


In [19]:
model = lasso_pipeline.named_steps["model"]

print("Iterations used:", model.n_iter_)
print("Maximum iterations:", model.max_iter)

Iterations used: 50000
Maximum iterations: 50000


> **Note:**  
> During training, the Lasso Regression model may display a `ConvergenceWarning`, indicating that the optimization algorithm reached the maximum number of iterations before fully satisfying the convergence criterion. This warning is related to the iterative optimization process used by Lasso and does not indicate an issue with the preprocessing pipeline or model implementation. Since this notebook focuses on comparing models using their default regularization settings, further optimization of the solver parameters and regularization strength is deferred to the hyperparameter tuning stage.

----

### 2.3 Generate Predictions


In [20]:
y_train_pred = lasso_pipeline.predict(X_train)
y_valid_pred = lasso_pipeline.predict(X_valid)

---

### 2.4 Model Evaluation

The performance of the trained Lasso Regression model is evaluated on both the training and validation datasets using the reusable regression evaluation function developed earlier in the project. The same evaluation metrics used for the previous models are applied to ensure a consistent and fair comparison.


In [21]:
n_features = len(
    lasso_pipeline.named_steps["column_transformer"]
    .get_feature_names_out()
)

lasso_train_metrics = evaluate_regression_model(
    y_true=y_train,
    y_pred=y_train_pred,
    n_features=n_features,
    adjusted_r2=True,
)

lasso_validation_metrics = evaluate_regression_model(
    y_true=y_valid,
    y_pred=y_valid_pred,
)

----

### 2.5 Performance Summary


In [22]:
lasso_model_evaluation_summary = pd.DataFrame(
    {
        "Training": lasso_train_metrics,
        "Validation": lasso_validation_metrics,
    }
)

lasso_model_evaluation_summary

,Training,Validation
MAE,11993.2646,19101.8859
MSE,307931334.0642,1487403595.5801
RMSE,17547.9724,38566.8717
R²,0.9522,0.7930
Adjusted R²,0.9302,NaN


---

In [23]:
import numpy as np

lasso_model = lasso_pipeline.named_steps["model"]

coef = lasso_model.coef_

print(f"Total coefficients: {len(coef)}")
print(f"Zero coefficients: {np.sum(coef == 0)}")
print(f"Non-zero coefficients: {np.sum(coef != 0)}")

Total coefficients: 294
Zero coefficients: 31
Non-zero coefficients: 263


### 2.6 Interpretation

The Lasso Regression model demonstrates a noticeable improvement over the baseline Linear Regression model on the validation dataset, indicating that L1 regularization helps improve generalization by reducing model complexity.

Compared with the baseline model, the validation error decreases substantially, and the validation R² score increases from **0.4430** to **0.7554**. This suggests that regularization reduces overfitting and enables the model to make more reliable predictions on unseen data.

However, the Lasso model does not perform as well as Ridge Regression. Although Lasso performs automatic feature selection by shrinking some coefficients exactly to zero, this sparsity may also remove features that still contain useful predictive information. For this dataset, retaining all features while shrinking their coefficients, as done by Ridge Regression, provides better predictive performance.

Overall, Lasso Regression improves upon the baseline model but is outperformed by Ridge Regression under the default hyperparameter settings. Further improvements may be achieved through hyperparameter tuning in a later stage of the project.


The trained Lasso Regression model assigned **35 out of 302 coefficients** exactly to zero. This indicates that the model automatically excluded these features from the prediction process, demonstrating Lasso's embedded feature selection capability. The remaining **267 features** retained non-zero coefficients and contributed to the final predictions.

----

## 3. ElasticNet Regression

### Overview

ElasticNet Regression is a regularized version of Linear Regression that combines both **L1 (Lasso)** and **L2 (Ridge)** regularization. By balancing coefficient shrinkage and feature selection, ElasticNet aims to improve generalization while addressing some of the limitations of using Lasso or Ridge individually.

In this section, the ElasticNet Regression model is trained using the same preprocessing pipeline as the previous models and evaluated using the same performance metrics to enable a fair comparison.

---

### Objectives

The objectives of this section are to:

- Train an ElasticNet Regression model using the preprocessed training dataset.
- Generate predictions for the training and validation datasets.
- Evaluate model performance using the reusable evaluation function.
- Compare its performance with the previously trained regression models.
---

### 3.1 Model Initialization

The ElasticNet Regression model is initialized using the `ElasticNet` estimator from scikit-learn with its default hyperparameters. Similar to the previous models, the estimator is integrated into the reusable machine learning pipeline to ensure a consistent preprocessing and training workflow.

Using the same pipeline for every model enables a fair comparison by ensuring that all models receive identical input data and preprocessing transformations.

In [24]:
from sklearn.linear_model import ElasticNet

elasticnet_pipeline = create_pipeline(
    ElasticNet()
)

----

### 3.2 Model Training

The initialized ElasticNet Regression pipeline is trained using the training dataset. During training, the preprocessing pipeline learns the required data transformations, and the ElasticNet Regression model learns the relationship between the input features and the target variable while applying both L1 and L2 regularization.

Training the complete pipeline ensures that all preprocessing steps and model fitting are performed as a single workflow, maintaining consistency between training and prediction.


In [25]:
elasticnet_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('dataset_preprocessing', ...), ('column_transformer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('numerical_missing_values', ...), ('categorical_missing_values', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
lotfrontage_median_,"Series[float64](25,)",Neighborhood ...dtype: float64
Name,Type,Value


---

### 3.3 Generate Predictions

After training, the pipeline is used to generate predictions for both the training and validation datasets. These predictions are subsequently used to evaluate the model's performance and compare it with the previously trained regression models.


In [26]:
y_train_pred = elasticnet_pipeline.predict(X_train)
y_valid_pred = elasticnet_pipeline.predict(X_valid)

---

### 3.4 Model Evaluation

The performance of the trained ElasticNet Regression model is evaluated on both the training and validation datasets using the reusable regression evaluation function developed earlier in the project. The same evaluation metrics are used to ensure a consistent and fair comparison across all models.

In [27]:
n_features = len(
    elasticnet_pipeline.named_steps["column_transformer"]
    .get_feature_names_out()
)

elasticnet_train_metrics = evaluate_regression_model(
    y_true=y_train,
    y_pred=y_train_pred,
    n_features=n_features,
    adjusted_r2=True,
)

elasticnet_validation_metrics = evaluate_regression_model(
    y_true=y_valid,
    y_pred=y_valid_pred,
)

---

### 3.5 Performance Summary

The evaluation metrics for the ElasticNet Regression model on the training and validation datasets are summarized below.

In [28]:
elasticnet_model_evaluation_summary = pd.DataFrame(
    {
        "Training": elasticnet_train_metrics,
        "Validation": elasticnet_validation_metrics,
    }
)

elasticnet_model_evaluation_summary

,Training,Validation
MAE,19250.2738,20175.4319
MSE,1120826544.9629,1412509098.9613
RMSE,33478.7477,37583.3620
R²,0.8260,0.8035
Adjusted R²,0.7459,NaN


----

### 3.6 Interpretation

The ElasticNet Regression model demonstrates a substantial improvement over the baseline Linear Regression model, indicating that combining L1 and L2 regularization helps improve the model's ability to generalize to unseen data.

Compared with the baseline model, the validation error decreases considerably, and the validation R² score increases from **0.4430** to **0.8328**. This suggests that the combination of coefficient shrinkage and feature selection effectively reduces overfitting while retaining strong predictive performance.

However, ElasticNet does not outperform Ridge Regression on this dataset. Although ElasticNet balances the advantages of Ridge and Lasso, the results indicate that retaining all features while shrinking their coefficients, as performed by Ridge Regression, provides better predictive performance for this problem.

Overall, ElasticNet offers a strong improvement over the baseline model and outperforms Lasso Regression under the default hyperparameter settings. However, Ridge Regression remains the best-performing model among the models evaluated in this notebook. Further improvements may be achieved through hyperparameter tuning in a later stage of the project.

----

## 4. Model Comparison

## 4.1 Performance Comparison

The performance of all regression models is summarized below. Each model was trained and evaluated using the same train-validation split, preprocessing pipeline, and evaluation metrics, ensuring a fair comparison.

The table below provides a side-by-side comparison of the baseline Linear Regression model and the regularized regression models.

In [29]:
comparison_results = pd.DataFrame(
    {
        "Linear Regression": baseline_validation_metrics,
        "Ridge Regression": ridge_validation_metrics,
        "Lasso Regression": lasso_validation_metrics,
        "ElasticNet Regression": elasticnet_validation_metrics,
    }
)

comparison_results

,Linear Regression,Ridge Regression,Lasso Regression,ElasticNet Regression
MAE,19473.3362,18882.8172,19101.8859,20175.4319
MSE,1511070187.3535,1132045883.0298,1487403595.5801,1412509098.9613
RMSE,38872.4863,33645.8895,38566.8717,37583.3620
R²,0.7898,0.8425,0.7930,0.8035


In [30]:
comparison_summary = pd.DataFrame(
    {
        "Train R²": [
            baseline_train_metrics["R²"],
            ridge_train_metrics["R²"],
            lasso_train_metrics["R²"],
            elasticnet_train_metrics["R²"],
        ],
        "Validation R²": [
            baseline_validation_metrics["R²"],
            ridge_validation_metrics["R²"],
            lasso_validation_metrics["R²"],
            elasticnet_validation_metrics["R²"],
        ],
        "Train RMSE": [
            baseline_train_metrics["RMSE"],
            ridge_train_metrics["RMSE"],
            lasso_train_metrics["RMSE"],
            elasticnet_train_metrics["RMSE"],
        ],
        "Validation RMSE": [
            baseline_validation_metrics["RMSE"],
            ridge_validation_metrics["RMSE"],
            lasso_validation_metrics["RMSE"],
            elasticnet_validation_metrics["RMSE"],
        ],
    },
    index=[
        "Linear Regression",
        "Ridge Regression",
        "Lasso Regression",
        "ElasticNet Regression",
    ],
)

comparison_summary

,Train R²,Validation R²,Train RMSE,Validation RMSE
Linear Regression,0.9522,0.7898,17542.2050,38872.4863
Ridge Regression,0.9331,0.8425,20765.3766,33645.8895
Lasso Regression,0.9522,0.7930,17547.9724,38566.8717
ElasticNet Regression,0.8260,0.8035,33478.7477,37583.3620


----

### 4.2 Model Interpretation
The comparison demonstrates the impact of regularization on the predictive performance of linear regression models.

Among all evaluated models, **Ridge Regression** achieved the best performance on the validation dataset, producing the lowest prediction errors and the highest coefficient of determination (R²). This indicates that applying L2 regularization effectively reduced overfitting while preserving the predictive contribution of all features.

**ElasticNet Regression** also showed a significant improvement over the baseline model by combining L1 and L2 regularization. However, its performance remained slightly below that of Ridge Regression.

**Lasso Regression** improved generalization compared with the baseline Linear Regression model and automatically removed less important features by assigning zero coefficients. Although this embedded feature selection reduced model complexity, it also discarded some predictive information, resulting in lower predictive performance than Ridge Regression.

Overall, the results indicate that shrinking feature coefficients without eliminating them is more suitable for this dataset than aggressively selecting a subset of features.

----

### 4.3 Model Selection



Based on the evaluation results, Ridge Regression achieved the best validation performance among all evaluated models. Therefore, it is selected as the primary candidate for further optimization.

ElasticNet Regression also demonstrated strong generalization performance, outperforming both the baseline Linear Regression and Lasso Regression models. Since its regularization strength (`alpha`) and the balance between L1 and L2 penalties (`l1_ratio`) have not yet been optimized, ElasticNet is also selected for hyperparameter tuning.